## **Experiment 07 (Variant):** Training and Performance Analysis of Feedforward Neural Networks using the Backpropagation Algorithm — **Wine Dataset**

**Course:** ICE 4111: Artificial Intelligence Lab  
**Program:** Bachelor of Science in Information and Communication Engineering (BSc in ICE)  
**Instructor:** Md. Ryhan Uddin

**Student Name:** ______________________________  
**Student ID:** ______________________________  
**Section/Batch:** ______________________________  
**Date of Submission:** ______________________________  


### **Quick Overview**

In this lab you will build a feedforward neural network with two hidden layers in TensorFlow/Keras and train it on the **Wine dataset** using the **backpropagation algorithm** with the Adam optimizer and early stopping. The original experiment used the Digits (image) dataset; here the same pipeline is applied to a **tabular chemical-analysis dataset**, so the input layer, feature scaling, and visualizations are adapted accordingly. You will track training through learning curves, evaluate the model with a full classification report and confusion matrix, and inspect the correct and misclassified predictions.

### **Learning Objectives**

After completing this experiment, students will be able to:

- Train a feedforward neural network for multiclass classification using backpropagation.
- Apply the Adam optimizer and early stopping to a training loop.
- Explain why tabular features with different units must be standardized, and why the scaler is fitted on training data only.
- Monitor training and validation curves to diagnose underfitting and overfitting.
- Evaluate a multiclass model using accuracy, a classification report, and a confusion matrix.
- Inspect misclassified samples and their prediction confidence to understand model behavior.


## **Theory**

### Introduction

A feedforward neural network passes information from the input layer through one or more hidden layers to the output layer without cycles. Training uses the **backpropagation algorithm** to compute how each weight contributes to the final error, and a gradient-based optimizer then updates the weights to reduce that error. This experiment focuses on the learning process itself: how loss and accuracy evolve over epochs.

### Mathematical Background

If the loss function is $L(\hat{y}, y)$ and the network output is obtained by composing layers, backpropagation uses the chain rule to compute gradients such as $\partial L / \partial W$. For a layer $z = Wa + b$ with activation $a' = \phi(z)$, the gradient of the loss is propagated backward from the output layer to earlier layers:

$$\frac{\partial L}{\partial W} = \frac{\partial L}{\partial a'} \cdot \frac{\partial a'}{\partial z} \cdot \frac{\partial z}{\partial W}$$

For multiclass classification, the output layer uses a **softmax** activation

$$\hat{y}_k = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

so that the outputs form a probability distribution over all classes, and the network is trained with **categorical cross-entropy loss**:

$$L = -\sum_{k=1}^{K} y_k \log(\hat{y}_k)$$

For tabular inputs we also use **standardization** (z-score scaling), which gives each feature zero mean and unit variance:

$$x' = \frac{x - \mu_{train}}{\sigma_{train}}$$

### Important Concepts

The practical training loop is: initialize weights, perform forward propagation, compute the loss, backpropagate gradients, update weights, and repeat for many epochs. **Early stopping** halts training when validation performance no longer improves for a set number of epochs (patience), and restores the weights from the best epoch.

Unlike pixel intensities (all in the same 0-16 range), Wine features have very different scales (for example *proline* is in the hundreds or thousands while *hue* is around 1). Without standardization, large-valued features dominate the gradients and slow down or destabilize training. The scaler must be **fitted on the training set only** and then applied to validation and test data; otherwise information from the test set leaks into training.

Labels are converted to **one-hot vectors** (e.g. class `2` becomes `[0,0,1]`) so they can be compared directly against the softmax output using categorical cross-entropy.

### Advantages and Applications

**Advantages**

- Feedforward networks are easy to build and are a strong baseline for classification problems.
- Backpropagation scales efficiently to networks with many layers and parameters.
- Hidden layers learn useful intermediate features automatically, without manual feature engineering.

**Disadvantages**

- On small tabular datasets, simpler models (logistic regression, random forest) are often just as good and less sensitive to hyperparameters.
- Training is sensitive to learning rate, layer sizes, initialization, and feature scaling.
- With very few samples (this dataset has only 178), results can vary noticeably between random splits and seeds.

**Applications**

Feedforward networks are directly useful for tabular data classification, such as identifying the origin of a product from its chemical measurements, as in this lab.

```mermaid
flowchart LR
    A[Input Layer: 13 Chemical Features] --> B[Hidden Layer 1: ReLU]
    B --> C[Hidden Layer 2: ReLU]
    C --> D[Output Layer: Softmax, 3 classes]
    D --> E[Categorical Cross-Entropy Loss]
    E --> F[Backpropagation]
    F --> B
    F --> C
```


## **Required Software and Libraries**

- Python 3.12 or later
- Jupyter Notebook, JupyterLab, VS Code, or Google Colab
- NumPy
- Pandas
- Matplotlib
- TensorFlow 2.x / Keras
- Scikit-learn

### Starter Check

Run the first code cell. If it prints `Setup complete. You can start the experiment.`, your environment is ready.

## **Dataset Description**

This experiment uses the **Wine dataset** from Scikit-learn. It is available locally (no download required) and is small enough to train in seconds.

- **Samples:** 178 wines grown in the same region of Italy.
- **Features:** 13 continuous chemical measurements (alcohol, malic acid, ash, alcalinity of ash, magnesium, total phenols, flavanoids, nonflavanoid phenols, proanthocyanins, color intensity, hue, OD280/OD315 of diluted wines, proline).
- **Target labels:** 3 classes (3 different cultivars): `class_0` (59 samples), `class_1` (71 samples), `class_2` (48 samples).
- **Source:** `sklearn.datasets.load_wine`.

## **Experimental Procedure**

1. Load the Wine dataset and inspect its features and class distribution.
2. Visualize how a few features and a 2-D PCA projection separate the classes.
3. Split the data into training, validation, and test sets (stratified).
4. Standardize the features using statistics from the training set only.
5. Convert labels to one-hot vectors.
6. Build a feedforward neural network with two hidden layers and a softmax output layer.
7. Train the model with early stopping and record the learning curves.
8. Evaluate test accuracy, the classification report, and the confusion matrix.
9. Inspect misclassified samples and the model's prediction confidence.

### Student Checkpoint

Before moving to the discussion section, make sure you can answer:

- Why must the `StandardScaler` be fitted on the training set only?
- At what epoch did early stopping halt training, and what does the gap (or lack of gap) between training and validation accuracy tell you?
- Which classes does the confusion matrix show being confused, and do the PCA plot and feature histograms explain why?


## **Source Code**

Run the following cells one by one. Read the output after each cell and add comments in your own words where instructed by your teacher.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress verbose TensorFlow startup logs

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(42)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

print("Setup complete. You can start the experiment.")

In [ ]:
# Load the Wine dataset

wine = load_wine()
X = wine.data.astype("float32")
y = wine.target
feature_names = list(wine.feature_names)
class_names = [str(n) for n in wine.target_names]
num_features = X.shape[1]
num_classes = len(class_names)

print("Shape:", X.shape)
print("Classes:", class_names)
print("Class counts:", np.bincount(y))
print("Missing values:", int(np.isnan(X).sum()))

df = pd.DataFrame(X, columns=feature_names)
df.describe().T[["mean", "std", "min", "max"]].round(2)

In [ ]:
# Visualize the data: feature histograms by class and a 2-D PCA projection

colors = ["#2a9d8f", "#e9c46a", "#e76f51"]
selected = ["alcohol", "flavanoids", "color_intensity", "proline"]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, feat in zip(axes[:4], selected):
    j = feature_names.index(feat)
    for c in range(num_classes):
        ax.hist(X[y == c, j], bins=15, alpha=0.6, color=colors[c], label=class_names[c])
    ax.set_title(feat)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")
axes[0].legend()

# PCA is done on standardized data just for visualization
X_vis = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))
for c in range(num_classes):
    axes[4].scatter(X_vis[y == c, 0], X_vis[y == c, 1], s=25, alpha=0.8, color=colors[c], label=class_names[c])
axes[4].set_title("PCA Projection (2 components)")
axes[4].set_xlabel("PC1")
axes[4].set_ylabel("PC2")

plt.suptitle("Wine Dataset: Class Separation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Train / Validation / Test Split (stratified)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full,
)
y_test_int = y_test

print("Train shape     :", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape      :", X_test.shape)

In [ ]:
# Standardize features (fit on TRAINING data only) and one-hot encode labels

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

y_train = tf.keras.utils.to_categorical(y_train_int, num_classes=num_classes)
y_val = tf.keras.utils.to_categorical(y_val_int, num_classes=num_classes)
y_test_oh = tf.keras.utils.to_categorical(y_test_int, num_classes=num_classes)

print("Train feature means (should be ~0):", np.round(X_train.mean(axis=0)[:5], 3), "...")
print("Train feature stds  (should be ~1):", np.round(X_train.std(axis=0)[:5], 3), "...")
print("Example one-hot label:", y_train[0], "<- class", y_train_int[0])

In [ ]:
# Build the Feedforward Neural Network

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(num_classes, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Train the Network with Early Stopping (Backpropagation)
# The dataset is small, so we allow more epochs and a larger patience than in the Digits experiment.

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=16,
    verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)],
)

epochs_run = len(history.history["loss"])
best_epoch = int(np.argmin(history.history["val_loss"])) + 1
print(f"Training finished after {epochs_run} epochs (best epoch by val_loss: {best_epoch}).")
print(f"Final train accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final val accuracy  : {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Evaluate on the Test Set

test_probabilities = model.predict(X_test, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)
test_true = np.argmax(y_test_oh, axis=1)

test_accuracy = accuracy_score(test_true, test_predictions)
print(f"Test Accuracy : {test_accuracy:.4f}")
print(classification_report(test_true, test_predictions, target_names=class_names, zero_division=0))

In [ ]:
# Training vs Validation Accuracy and Loss Curves

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history["accuracy"], label="Train", color="#2a9d8f")
axes[0].plot(history.history["val_accuracy"], label="Validation", color="#e76f51")
axes[0].set_title("Accuracy Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train", color="#2a9d8f")
axes[1].plot(history.history["val_loss"], label="Validation", color="#e76f51")
axes[1].axvline(best_epoch - 1, color="gray", linestyle="--", alpha=0.7, label="Best epoch")
axes[1].set_title("Loss Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Categorical Cross-Entropy Loss")
axes[1].legend()

plt.suptitle("Feedforward Network Learning Curves (Wine)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix

cm = confusion_matrix(test_true, test_predictions, labels=list(range(num_classes)))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap="Blues", ax=ax, colorbar=True, values_format="d")

plt.title("Confusion Matrix: Wine Cultivar Classification")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
# Per-Class Accuracy

per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(6, 4.5))
plt.bar(class_names, per_class_accuracy, color=["#2a9d8f", "#e9c46a", "#e76f51"])

plt.xlabel("Class")
plt.ylabel("Accuracy")
plt.title("Per-Class Test Accuracy")
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.show()

In [ ]:
# Inspect Correct and Misclassified Predictions (with prediction confidence)

misclassified = np.where(test_predictions != test_true)[0]
correct = np.where(test_predictions == test_true)[0]

print(f"Correctly classified : {len(correct)} / {len(test_true)}")
print(f"Misclassified         : {len(misclassified)} / {len(test_true)}")

confidence = test_probabilities.max(axis=1)

if len(misclassified) > 0:
    wrong_df = pd.DataFrame({
        "test_index": misclassified,
        "true_class": [class_names[i] for i in test_true[misclassified]],
        "predicted_class": [class_names[i] for i in test_predictions[misclassified]],
        "confidence_in_prediction": confidence[misclassified].round(3),
        "prob_of_true_class": test_probabilities[misclassified, test_true[misclassified]].round(3),
    })
    display(wrong_df)
else:
    print("No misclassified samples on the test set with this seed/split.")

plt.figure(figsize=(8, 4.5))
plt.hist(confidence[correct], bins=10, alpha=0.7, color="green", label="Correct")
if len(misclassified) > 0:
    plt.hist(confidence[misclassified], bins=10, alpha=0.7, color="red", label="Misclassified")
plt.xlabel("Model confidence (max softmax probability)")
plt.ylabel("Number of test samples")
plt.title("Prediction Confidence: Correct vs Misclassified")
plt.legend()
plt.show()

## **Expected Output**

After running all cells, the notebook should display:

- Dataset shape (178 x 13), class names, class counts, and a feature summary table.
- Feature histograms by class and a 2-D PCA scatter plot.
- Train/validation/test split shapes and a check that the standardized features have mean ~0 and std ~1.
- The network summary and the epoch at which early stopping triggered.
- Test accuracy and a full classification report across the 3 wine classes.
- Accuracy and loss learning curves.
- A confusion matrix and a per-class accuracy bar chart.
- A table of misclassified samples (if any) and a confidence histogram.

Because the test set contains only 36 samples, each single error changes the accuracy by about 2.8 percentage points. Exact numbers can differ slightly between machines and TensorFlow versions even with a fixed seed.

You should include the important outputs in your lab report and briefly explain what each output means.

## **Performance Evaluation**

The most relevant metrics are training accuracy, validation accuracy, test accuracy, and the shapes of the learning curves. A small, stable gap between training and validation performance usually indicates good generalization, while a widening gap suggests overfitting. With such a small dataset, also pay attention to how noisy the validation curve is. The confusion matrix and the misclassification table show whether errors are concentrated on specific classes and whether the model was confident when it was wrong.

## **Discussion**

Backpropagation efficiently computes how every weight contributed to the final error, so gradient descent can update all layers at once. On the Wine dataset the classes are fairly well separated in feature space (see the PCA plot), so even a small network reaches high accuracy quickly. Compared with the Digits experiment, the key differences are: the inputs need standardization because features have different units, the dataset is much smaller (so training/validation curves are noisier and a larger early-stopping patience helps), and there is no spatial structure for the network to exploit, so a simpler model such as logistic regression would likely perform similarly.

## **Viva Questions**

1. **Q:** What is a feedforward neural network?  
   **A:** A network where data flows from the input layer to the output layer through one or more hidden layers without cycles.

2. **Q:** What does backpropagation compute?  
   **A:** The gradients of the loss with respect to every weight in the network, using the chain rule.

3. **Q:** Why is the Adam optimizer popular?  
   **A:** It combines momentum with adaptive, per-parameter learning rates, which usually leads to fast and stable convergence.

4. **Q:** Why do we standardize the Wine features, and why fit the scaler on the training set only?  
   **A:** Features have very different scales, which distorts gradient updates. Fitting on training data only prevents information from the validation/test sets leaking into training.

5. **Q:** Why is one-hot encoding used for the labels here?  
   **A:** Because the output layer uses softmax with categorical cross-entropy, which compares a probability distribution against a one-hot target vector.

6. **Q:** What is early stopping, and why use it?  
   **A:** Stopping training when validation performance stops improving for a set number of epochs, to prevent overfitting and save training time.

7. **Q:** Why did the input and output layer sizes change compared with the Digits experiment?  
   **A:** The input layer must match the number of features (13 instead of 64) and the output layer must match the number of classes (3 instead of 10).

8. **Q:** What is the role of the hidden layers?  
   **A:** To learn intermediate feature representations from the raw inputs that make the classes easier to separate.

9. **Q:** What does a rising validation loss alongside falling training loss indicate?  
   **A:** Overfitting: the model is fitting the training data at the expense of generalization.

10. **Q:** Why is a confusion matrix useful beyond overall accuracy?  
    **A:** It shows exactly which classes are being confused with each other, which a single accuracy number cannot reveal.

## **Exercises**

1. Change the number of hidden units (e.g. 32/16 instead of 64/32) and compare the test accuracy.
2. Try `tanh` activation instead of `relu` in the hidden layers and observe the effect on convergence speed.
3. Add a `Dropout` layer after each hidden layer and check whether validation loss improves or the train/validation gap narrows.
4. Train the network without early stopping (set a fixed number of epochs) and compare the learning curves.
5. Remove the `StandardScaler` step (feed raw features) and compare convergence and accuracy. What do you observe?
6. Repeat the experiment with 5 different random seeds / splits and report the mean and standard deviation of test accuracy.

## **Lab Report Submission Checklist**

- Completed cover information.
- Objective and theory summary in your own words.
- All important code outputs, including the classification report.
- The feature histograms/PCA plot, learning curves, confusion matrix, per-class accuracy chart, and misclassification table.
- Short discussion of which classes were confused (if any) and why.
- Answers to assigned viva questions.
- Completed exercises.

**Reminder:** A good lab report should not only show screenshots. It should explain what the outputs mean.